Data Aggregation Stuff

Kyle - Given a latitude and longitude and a radius and a set of points, find every point in that radius and turn that into the individual datasets

the idea is make some python functions which do this, geopandas might be the right library to use?
this will enable the joining of our many datasets: each row in our final dataset would be e.g. one sensor location with the aggregate stat for each associated dataset within that radius - so sum of building fuel use, sum of construction permits, sum of traffic (you don't have to do the aggregation stuff for now since we'll need to think more about the right way to do it)


In [1]:
# STANDARDIZATION

STANDARD_COLUMNS = [
    "source",
    "sensor_id",
    "latitude",
    "longitude",
    "year",
    "month",
    "pm25"
]

def standardize_sensor_data(df, source, column_map):
    # Rename columns
    result = df.rename(columns=column_map).copy()

    # Add data source
    result["source"] = source

    # 4. Only keep the standardized columns
    result = result[STANDARD_COLUMNS]

    # 5. Standardize data types
    result["sensor_id"] = result["sensor_id"].astype("string")

    for col in ["latitude", "longitude", "pm25"]:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        )

    for col in ["year", "month"]:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        ).astype("Int64")

    # 6. Remove invalid coordinates
    result = result.dropna(
        subset=["sensor_id", "latitude", "longitude"]
    )

    # 7. Validate coordinates and dates
    valid = (
        result["latitude"].between(-90, 90) &
        result["longitude"].between(-180, 180) &
        result["month"].between(1, 12) &
        result["year"].notna()
    )

    result = result.loc[valid.fillna(False)].copy()

    # 8. Create source-qualified sensor ID
    result["sensor_key"] = (
        result["source"] + "_" + result["sensor_id"]
    )

    return result


epa_col_map =  {
    "Site Num": "sensor_id",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Year": "year",
    "Month": "month",
    "Arithmetic Mean": "pm25"
}
openaq_col_map =  {
    "location_id": "sensor_id",
    "latitude": "latitude",
    "longitude": "longitude",
    "year": "year",
    "month": "month",
    "pm25_mean_ug_m3": "pm25"
}
purple_col_map =  {
    "sensor_index": "sensor_id",
    "latitude": "latitude",
    "longitude": "longitude",
    "Year": "year",
    "Month": "month",
    "pm25_clean": "pm25"
}

In [2]:
import pandas as pd
import geopandas as gpd

# Find data for all of our features: greenery, traffic, boiler, building, etc.
# Convert each dataset to a geopandas dataframe
# Spatially join on our sensor data using a radius
# Aggregate the values of the records that are within the sensor's radius
# Now we have a aggregate dataset with sensor --> aggregate levels of our features
# Can create ML model for this!


def to_gdf(df, lat_col="latitude", lon_col="longitude"):
    df = df.dropna(subset=[lat_col, lon_col]).copy()
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    )
    return gdf.to_crs("EPSG:32618")


def aggregate_near_sensors(
    sensors_gdf,
    features_gdf,
    prefix,
    radius=1609.344,
    mode="monthly",
    year_col=None,
    month_col=None,
    agg_cols=None,
    agg_type="sum"
):
    # Ensure same CRS
    features_gdf = features_gdf.to_crs(sensors_gdf.crs)

    # STEP 1: Prepare temporal columns
    if mode == "monthly":
        features_gdf = features_gdf.rename(
            columns={
                year_col: "feature_year",
                month_col: "feature_month"
            }
        )
        time_cols = ["feature_year", "feature_month"]

    elif mode == "yearly":
        features_gdf = features_gdf.rename(
            columns={year_col: "feature_year"}
        )
        time_cols = ["feature_year"]

    elif mode == "interval":
        time_cols = ["issue_ym", "expiration_ym"] # pre-processed to improve speed

    elif mode == "static":
        time_cols = []

    else:
        raise ValueError("Invalid mode")

    # STEP 2: Determine columns needed for aggregation
    if agg_cols is None:
        agg_cols = {}

    join_cols = list(dict.fromkeys(
        [*time_cols, *agg_cols.keys(), "geometry"]
    ))

    # STEP 3: Spatial join
    nearby = gpd.sjoin(
        sensors_gdf,
        features_gdf[join_cols],
        how="inner",
        predicate="dwithin",
        distance=radius
    )

    # STEP 4: Temporal filtering
    if mode == "monthly":
        nearby = nearby[
            (nearby["year"] == nearby["feature_year"]) &
            (nearby["month"] == nearby["feature_month"])
        ]

    elif mode == "yearly":
        nearby = nearby[
            nearby["year"] == nearby["feature_year"]
        ]

    elif mode == "interval":
        nearby = nearby[
            (nearby["sensor_ym"] >= nearby["issue_ym"]) &
            (nearby["sensor_ym"] <= nearby["expiration_ym"])
        ]

    # Static mode: no temporal filtering

    # STEP 5: Aggregate
    group_cols = ["sensor_key", "year", "month"]

    if agg_type == "count":
        aggregated = (
            nearby.groupby(group_cols)
            .size()
            .reset_index(name=f"{prefix}_count")
        )

    elif agg_type in ["sum", "mean"]:
        aggregated = (
            nearby.groupby(group_cols)
            .agg(
                **{
                    f"{prefix}_{col}_{stat}": (col, stat)
                    for col, stat in agg_cols.items()
                }
            )
            .reset_index()
        )

    else:
        raise ValueError("agg_type must be 'sum', 'mean', or 'count'")

    return aggregated

In [3]:
# For building data

numeric_cols = [
    "Fuel Oil #2 Use (kBtu)",
    "Natural Gas Use (kBtu)",
    "Fuel Oil #4 Use (kBtu)",
    "Direct GHG Emissions (Metric Tons CO2e)",
    "Property GFA - Calculated (Buildings) (ft²)",
    "Site EUI (kBtu/ft²)"
]

common_cols = [
    "Latitude",
    "Longitude",
    *numeric_cols
]

# Load modern data
building_data_df = pd.read_csv(
    "Datas/buildings_data.csv",
    usecols=["Calendar Year", *common_cols],
    low_memory=False
)

# Load historical data
historical_dfs = []

for year in range(2016, 2022):
    b = pd.read_csv(
        f"Datas/LocalLaw84_{year}.csv",
        usecols=["Year Ending", *common_cols],
        low_memory=False
    )

    b["Calendar Year"] = pd.to_datetime(
        b["Year Ending"],
        format="%m/%d/%Y",
        errors="coerce"
    ).dt.year

    b = b.drop(columns=["Year Ending"])

    historical_dfs.append(b)

# Combine all years
building_data_df = pd.concat(
    [building_data_df, *historical_dfs],
    ignore_index=True
)

# Convert numeric measurements
for col in numeric_cols:
    building_data_df[col] = pd.to_numeric(
        building_data_df[col]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

# Ensure coordinates and years are numeric
for col in ["Calendar Year", "Latitude", "Longitude"]:
    building_data_df[col] = pd.to_numeric(
        building_data_df[col],
        errors="coerce"
    )

In [4]:
building_data_df["Calendar Year"].value_counts().sort_index()
building_data_df.groupby("Calendar Year")[numeric_cols].count()

,Fuel Oil #2 Use (kBtu),Natural Gas Use (kBtu),Fuel Oil #4 Use (kBtu),Direct GHG Emissions (Metric Tons CO2e),Property GFA - Calculated (Buildings) (ft²),Site EUI (kBtu/ft²)
Calendar Year,,,,,,
2016,3788,14370,1769,16216,16370,16095
2017,5765,29114,2388,33445,34655,32208
2018,4451,22691,1638,24957,25240,24731
2019,5101,26475,1744,28839,29072,28295
2020,4740,25524,1578,27879,28060,27228
2021,4483,25901,1438,29249,29840,27861
2022,4862,26936,1427,30329,30485,28890
2023,5056,29174,1358,32607,33672,30888
2024,4829,28889,1035,29066,39055,30577


In [5]:
boiler_df = pd.read_csv("Datas/boiler_permits.csv", thousands=',') # Some volumes contain commas
building_permit_df = pd.read_csv("Datas/DOB_building_permits.csv")[['BOROUGH', 'Issuance Date', 'Expiration Date','LATITUDE', 'LONGITUDE', 'Bin #', 'Lot']]
traffic_df = pd.read_csv("Datas/traffic.csv", thousands=',')
tree_df = pd.read_csv('Datas/tree_data.csv')[['address', 'borough', 'latitude', 'longitude', 'bin']]

# Our sensors: We gotta standardize these lil goonies.
epa_df = pd.read_csv("Datas/epa_output.csv")
openaq_df = pd.read_csv("Datas/openaq_top5_highest_pm25_by_month.csv")
purpleair_df = pd.read_csv("Datas/purple_air_monthlypm25.csv")

/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_69351/1307844936.py:2: DtypeWarning: Columns (8: Block, 9: Lot, 10: Community Board, 31: Permittee's Phone #, 33: Permittee's License #, 36: HIC License, 47: Owner's House #, 48: Owner's House Street Name, 49: Owner’s House City, 50: Owner’s House State, 51: Owner’s House Zip Code) have mixed types. Specify dtype option on import or set low_memory=False.
  building_permit_df = pd.read_csv("Datas/DOB_building_permits.csv")[['BOROUGH', 'Issuance Date', 'Expiration Date','LATITUDE', 'LONGITUDE', 'Bin #', 'Lot']]


In [6]:
building_data_df.dtypes

Calendar Year                                    int64
Site EUI (kBtu/ft²)                            Float64
Fuel Oil #2 Use (kBtu)                         Float64
Fuel Oil #4 Use (kBtu)                         Float64
Natural Gas Use (kBtu)                         Float64
Direct GHG Emissions (Metric Tons CO2e)        Float64
Property GFA - Calculated (Buildings) (ft²)    Float64
Latitude                                       float64
Longitude                                      float64
dtype: object

In [7]:
# Pre-Processing


# Change boiler issue and expiration date to dates
boiler_df['ExpirationDate_M'] = pd.to_datetime(boiler_df['ExpirationDate']).dt.month
boiler_df['ExpirationDate_Y'] = pd.to_datetime(boiler_df['ExpirationDate']).dt.year
boiler_df['IssueDate_M'] = pd.to_datetime(boiler_df['IssueDate']).dt.month
boiler_df['IssueDate_Y'] = pd.to_datetime(boiler_df['IssueDate']).dt.year

boiler_df["issue_ym"] = (
    boiler_df["IssueDate_Y"] * 12
    + boiler_df["IssueDate_M"]
)

boiler_df["expiration_ym"] = (
    boiler_df["ExpirationDate_Y"] * 12
    + boiler_df["ExpirationDate_M"]
)


# Do the same for building permits
building_permit_df["Expiration Date"] = pd.to_datetime(
    building_permit_df["Expiration Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

building_permit_df["Issuance Date"] = pd.to_datetime(
    building_permit_df["Issuance Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

building_permit_df["ExpirationDate_M"] = building_permit_df["Expiration Date"].dt.month
building_permit_df["ExpirationDate_Y"] = building_permit_df["Expiration Date"].dt.year

building_permit_df["IssueDate_M"] = building_permit_df["Issuance Date"].dt.month
building_permit_df["IssueDate_Y"] = building_permit_df["Issuance Date"].dt.year

building_permit_df["issue_ym"] = (
    building_permit_df["IssueDate_Y"] * 12
    + building_permit_df["IssueDate_M"]
)

building_permit_df["expiration_ym"] = (
    building_permit_df["ExpirationDate_Y"] * 12
    + building_permit_df["ExpirationDate_M"]
)


In [8]:
# print(len(building_permit_df['Expiration Date']))
# len(building_permit_df[~ building_permit_df['Expiration Date'].isna()])

In [9]:
building_data_gdf = to_gdf(building_data_df, "Latitude", "Longitude")
boiler_gdf = to_gdf(boiler_df, "latitude", "longitude")
building_permit_gdf = to_gdf(building_permit_df, "LATITUDE", "LONGITUDE")
traffic_gdf = to_gdf(traffic_df, 'latitude', 'longitude')
tree_gdf = to_gdf(tree_df, "latitude", "longitude")

In [10]:
tree_gdf.crs

<Projected CRS: EPSG:32618>
Name: WGS 84 / UTM zone 18N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 78°W and 72°W, northern hemisphere between equator and 84°N, onshore and offshore. Bahamas. Canada - Nunavut; Ontario; Quebec. Colombia. Cuba. Ecuador. Greenland. Haiti. Jamaica. Panama. Turks and Caicos Islands. United States (USA). Venezuela.
- bounds: (-78.0, 0.0, -72.0, 84.0)
Coordinate Operation:
- name: UTM zone 18N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [11]:
epa_df = standardize_sensor_data(epa_df, "EPA", epa_col_map)
openaq_df = standardize_sensor_data(openaq_df, "OPENAQ", openaq_col_map)
purpleair_df = standardize_sensor_data(purpleair_df, "PURPLEAIR", purple_col_map)

epa_gdf = to_gdf(epa_df, "latitude", "longitude")
openaq_gdf = to_gdf(openaq_df, "latitude", "longitude")
purpleair_gdf = to_gdf(purpleair_df, "latitude", "longitude")

sensors_gdf = gpd.GeoDataFrame(
    pd.concat([epa_gdf, openaq_gdf, purpleair_gdf], ignore_index=True),
    geometry="geometry", crs="EPSG:32618"
)

# Apply the same official borough boundary rule to EPA and OpenAQ sensors,
# not only PurpleAir. This also protects against older unfiltered CSVs.
boroughs = gpd.read_file(
    "https://data.cityofnewyork.us/resource/gthc-hcne.geojson?$limit=10"
).to_crs(sensors_gdf.crs)

before_count = len(sensors_gdf)
sensors_gdf = gpd.sjoin(
    sensors_gdf,
    boroughs[["geometry"]],
    how="inner",
    predicate="intersects"
)
# Remove only duplicated spatial matches (e.g., a boundary point),
# not distinct observations that happen to share a sensor-month key.
sensors_gdf = (
    sensors_gdf.loc[~sensors_gdf.index.duplicated(keep="first")]
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)
print(f"NYC sensor-month rows retained: {len(sensors_gdf)} / {before_count}")

sensors_gdf["sensor_ym"] = (
    sensors_gdf["year"] * 12
    + sensors_gdf["month"]
)

sensors_gdf.head(10)




NYC sensor-month rows retained: 3196 / 3196


,source,sensor_id,latitude,longitude,year,month,pm25,sensor_key,geometry,sensor_ym
0,EPA,110,40.8160,-73.90200,2016,1,6.624461,EPA_110,POINT (592600.643 4518911.64),24193
1,EPA,133,40.8679,-73.87809,2016,1,8.990909,EPA_133,POINT (594543.361 4524698.573),24193
2,EPA,110,40.8160,-73.90200,2017,1,6.758297,EPA_110,POINT (592600.643 4518911.64),24205
3,EPA,133,40.8679,-73.87809,2017,1,8.790909,EPA_133,POINT (594543.361 4524698.573),24205
4,EPA,110,40.8160,-73.90200,2018,1,7.876813,EPA_110,POINT (592600.643 4518911.64),24217
5,EPA,133,40.8679,-73.87809,2018,1,11.110000,EPA_133,POINT (594543.361 4524698.573),24217
6,EPA,110,40.8160,-73.90200,2019,1,7.226107,EPA_110,POINT (592600.643 4518911.64),24229
7,EPA,133,40.8679,-73.87809,2019,1,7.420000,EPA_133,POINT (594543.361 4524698.573),24229
8,EPA,110,40.8160,-73.90200,2020,1,8.254894,EPA_110,POINT (592600.643 4518911.64),24241
9,EPA,133,40.8679,-73.87809,2020,1,7.245455,EPA_133,POINT (594543.361 4524698.573),24241


PurpleAir A/B channel quality diagnostic (API spot check)
The imported purple_air_monthlypm25.csv contains pre-averaged PM2.5, so it cannot tell us whether channel A or channel B is faulty. This diagnostic finds the highest monthly PurpleAir values, retrieves the daily A and B channels for their months, and compares them. It does not modify the underlying data or remove observations.

Before running: Set PURPLEAIR_API_KEY in your environment (or a local .env file with python-dotenv installed). This makes up to five historical API requests and may consume PurpleAir API points. Change TOP_N to reduce requests. Check channel_diagnostics and channel_daily below.

The screening rule is exploratory: A/B disagreement is flagged when the absolute difference is greater than both 5 µg/m³ and 61% of their mean. It should be interpreted with the concentrations and time series rather than treated as automatic proof that a specific channel is broken.

From Kyle: ChatGPT, you're the best!

In [12]:
# Diagnostic only: inspect PurpleAir channel A vs B for suspicious sensor-months.
# Does NOT change purpleair_df, sensors_gdf, or final_df.
import os
import requests
import numpy as np
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # Set PURPLEAIR_API_KEY in your shell/environment instead.

API_KEY_DIAG = os.getenv("PURPLEAIR_API_KEY")
if not API_KEY_DIAG:
    raise RuntimeError(
        "Set PURPLEAIR_API_KEY (e.g., in a local .env file) before running this diagnostic."
    )

TOP_N = 5  # Limit API requests (one per selected sensor-month).
SUSPECT_THRESHOLD = 1000  # Only used to choose candidates, NOT to delete data.

# purpleair_df is already standardized in the previous cells (sensor_id, year, month, pm25).
candidates = (
    purpleair_df.loc[purpleair_df["pm25"] > SUSPECT_THRESHOLD,
                     ["sensor_id", "year", "month", "pm25"]]
    .dropna(subset=["sensor_id", "year", "month"])
    .sort_values("pm25", ascending=False)
    .drop_duplicates(subset=["sensor_id"])  # Worst month from each suspicious sensor.
    .head(TOP_N)
)
print("Selected suspicious sensor-months:")
print(candidates.to_string(index=False))


def fetch_purpleair_channels(sensor_id, year, month):
    """Download daily ATM channel measurements for one UTC calendar month."""
    start = pd.Timestamp(year=int(year), month=int(month), day=1, tz="UTC")
    end = start + pd.DateOffset(months=1)
    url = f"https://api.purpleair.com/v1/sensors/{int(sensor_id)}/history"
    response = requests.get(
        url,
        headers={"X-API-Key": API_KEY_DIAG},
        params={
            "start_timestamp": int(start.timestamp()),
            "end_timestamp": int(end.timestamp()),
            "average": 1440,
            "fields": "pm2.5_atm,pm2.5_atm_a,pm2.5_atm_b",
        },
        timeout=45,
    )
    response.raise_for_status()
    payload = response.json()
    fields = payload.get("fields", [])
    raw = payload.get("data") or []
    if not raw:
        return pd.DataFrame()
    daily = pd.DataFrame(raw, columns=fields)
    required = {"pm2.5_atm_a", "pm2.5_atm_b"}
    if not required.issubset(daily.columns):
        raise ValueError(f"Missing A/B fields for sensor {sensor_id}; received: {list(daily.columns)}")
    for col in ["pm2.5_atm", "pm2.5_atm_a", "pm2.5_atm_b"]:
        if col in daily:
            daily[col] = pd.to_numeric(daily[col], errors="coerce")
    if "time_stamp" in daily:
        daily["date_utc"] = pd.to_datetime(daily["time_stamp"], unit="s", utc=True, errors="coerce")
        # Some APIs return calendar-boundary records outside the requested month.
        daily = daily[daily["date_utc"].ge(start) & daily["date_utc"].lt(end)].copy()
    daily["sensor_id"] = str(sensor_id)
    daily["year"] = int(year)
    daily["month"] = int(month)
    avg = (daily["pm2.5_atm_a"] + daily["pm2.5_atm_b"]) / 2
    daily["ab_mean"] = avg
    daily["ab_abs_diff"] = (daily["pm2.5_atm_a"] - daily["pm2.5_atm_b"]).abs()
    # PurpleAir 24h A/B agreement screen: fail if BOTH thresholds exceeded.
    daily["ab_disagree"] = (
        daily["pm2.5_atm_a"].notna() & daily["pm2.5_atm_b"].notna() &
        (daily["ab_abs_diff"] > 5) & (daily["ab_abs_diff"] > 0.61 * avg)
    )
    return daily

all_daily = []
summary_rows = []
for r in candidates.itertuples(index=False):
    try:
        daily = fetch_purpleair_channels(r.sensor_id, r.year, r.month)
        if daily.empty:
            print(f"Sensor {r.sensor_id}, {r.year}-{int(r.month):02d}: no historical data returned")
            continue
        both = daily["pm2.5_atm_a"].notna() & daily["pm2.5_atm_b"].notna()
        summary_rows.append({
            "sensor_id": r.sensor_id,
            "year": int(r.year),
            "month": int(r.month),
            "previous_monthly_pm25": r.pm25,
            "valid_ab_days": int(both.sum()),
            "a_mean": daily["pm2.5_atm_a"].mean(),
            "b_mean": daily["pm2.5_atm_b"].mean(),
            "api_atm_mean": daily["pm2.5_atm"].mean() if "pm2.5_atm" in daily else np.nan,
            "disagree_days": int(daily.loc[both, "ab_disagree"].sum()),
            "disagree_fraction": daily.loc[both, "ab_disagree"].mean() if both.any() else np.nan,
            "max_ab_difference": daily["ab_abs_diff"].max(),
        })
        all_daily.append(daily)
    except (requests.RequestException, ValueError) as exc:
        print(f"Could not inspect sensor {r.sensor_id}, {r.year}-{int(r.month):02d}: {exc}")

channel_diagnostics = pd.DataFrame(summary_rows)
channel_daily = pd.concat(all_daily, ignore_index=True) if all_daily else pd.DataFrame()
print("\nA/B diagnostic summary (inspect A vs B and disagreement fractions):")
if not channel_diagnostics.empty:
    display(channel_diagnostics.round(2))
    print("\nDaily A/B readings for the worst selected sensor-month:")
    display(channel_daily.loc[
        channel_daily["sensor_id"] == str(channel_diagnostics.iloc[0]["sensor_id"]),
        [c for c in ["date_utc", "pm2.5_atm", "pm2.5_atm_a", "pm2.5_atm_b", "ab_abs_diff", "ab_disagree"]
         if c in channel_daily.columns]
    ].head(35))
else:
    print("No results. Check your API key, API permissions/credits, and candidate months.")


Selected suspicious sensor-months:
Empty DataFrame
Columns: [sensor_id, year, month, pm25]
Index: []

A/B diagnostic summary (inspect A vs B and disagreement fractions):
No results. Check your API key, API permissions/credits, and candidate months.


In [13]:
traffic_gdf

,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction,longitude,latitude,geometry
0,12512,Queens,2013,3,7,4,15,5,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB,-73.815732,40.674568,POINT (600088.093 4503306.242)
1,12512,Queens,2013,3,7,4,30,8,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB,-73.815732,40.674568,POINT (600088.093 4503306.242)
2,12512,Queens,2013,3,7,4,45,8,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB,-73.815732,40.674568,POINT (600088.093 4503306.242)
3,12512,Queens,2013,3,7,5,0,7,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB,-73.815732,40.674568,POINT (600088.093 4503306.242)
4,12512,Queens,2013,3,7,5,15,9,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB,-73.815732,40.674568,POINT (600088.093 4503306.242)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1875149,1286,Queens,2011,9,2,4,45,16,116319,POINT (1007135.1 201393),PAGE PL,MASPETH AV,GRAND AV,SB,-73.917442,40.719424,POINT (591430.599 4508174.774)
1875150,1286,Queens,2011,9,2,5,0,30,116319,POINT (1007135.1 201393),PAGE PL,MASPETH AV,GRAND AV,SB,-73.917442,40.719424,POINT (591430.599 4508174.774)
1875151,1286,Queens,2011,9,2,5,15,31,116319,POINT (1007135.1 201393),PAGE PL,MASPETH AV,GRAND AV,SB,-73.917442,40.719424,POINT (591430.599 4508174.774)
1875152,1286,Queens,2011,9,2,5,30,42,116319,POINT (1007135.1 201393),PAGE PL,MASPETH AV,GRAND AV,SB,-73.917442,40.719424,POINT (591430.599 4508174.774)


In [14]:
building_data_gdf.dtypes

Calendar Year                                     int64
Site EUI (kBtu/ft²)                             Float64
Fuel Oil #2 Use (kBtu)                          Float64
Fuel Oil #4 Use (kBtu)                          Float64
Natural Gas Use (kBtu)                          Float64
Direct GHG Emissions (Metric Tons CO2e)         Float64
Property GFA - Calculated (Buildings) (ft²)     Float64
Latitude                                        float64
Longitude                                       float64
geometry                                       geometry
dtype: object

In [15]:
# Time to Aggregate 

# parameters for function
    # sensors_gdf,
    # features_gdf,
    # prefix,
    # radius=1609.344,
    # mode="monthly",
    # year_col=None,
    # month_col=None,
    # agg_cols=None,
    # agg_type="sum"


boiler_agg = aggregate_near_sensors(
    sensors_gdf,
    boiler_gdf,
    prefix="boiler",
    mode="interval",
    agg_type="count"
)

traffic_agg = aggregate_near_sensors(
    sensors_gdf,
    traffic_gdf,
    prefix='traffic',
    year_col='Yr',
    month_col='M',
    mode='monthly',
    agg_cols={'Vol': 'sum'}
)


building_cols = {
      "Fuel Oil #2 Use (kBtu)" : "sum",
      "Fuel Oil #4 Use (kBtu)" : "sum",
      "Natural Gas Use (kBtu)" :  "sum",
      "Direct GHG Emissions (Metric Tons CO2e)" :  "sum",
      "Property GFA - Calculated (Buildings) (ft²)" :  "sum",
      "Site EUI (kBtu/ft²)" : "mean"
}

# building_data_gdf['Calendar Year'] = pd.to_numeric(building_data_gdf['Calendar Year'])

building_data_agg = aggregate_near_sensors(
    sensors_gdf,
    building_data_gdf,
    prefix='buildings',
    year_col='Calendar Year',
    mode='yearly',
    agg_cols=building_cols)

building_permit_agg = aggregate_near_sensors(
    sensors_gdf,
    building_permit_gdf,
    prefix='building_permit',
    mode='interval',
    agg_type='count'
)

tree_agg = aggregate_near_sensors(
    sensors_gdf,
    tree_gdf,
    prefix='tree',
    mode='static',
    agg_type='count'
)



In [ ]:
final_df = (
    sensors_gdf
    .merge(traffic_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(boiler_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(building_data_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(building_permit_agg, on=["sensor_key", "year", "month"], how="left")
    .merge(tree_agg, on=["sensor_key", "year", "month"], how="left")
)

# final_df.head(20)
# final_df.to_csv('final_data.csv', index=False)
final_df['pm25'].max()

# final_df["sensor_key"].unique()
# final_df.loc[final_df["tree_count"].isna(), "sensor_key"].unique()




,source,sensor_id,latitude,longitude,year,month,pm25,sensor_key,geometry,sensor_ym,traffic_Vol_sum,boiler_count,buildings_Fuel Oil #2 Use (kBtu)_sum,buildings_Fuel Oil #4 Use (kBtu)_sum,buildings_Natural Gas Use (kBtu)_sum,buildings_Direct GHG Emissions (Metric Tons CO2e)_sum,buildings_Property GFA - Calculated (Buildings) (ft²)_sum,buildings_Site EUI (kBtu/ft²)_mean,building_permit_count,tree_count
0,EPA,110,40.81600,-73.90200,2016,1,6.624461,EPA_110,POINT (592600.643 4518911.64),24193,227163.0,35,118466001.5,114223766.9,1357306558.0,90229.8,32138336.0,80.822015,1382,9106
1,EPA,133,40.86790,-73.87809,2016,1,8.990909,EPA_133,POINT (594543.361 4524698.573),24193,NaN,19,350557066.2,428966604.2,2054432883.5,178596.4,37925687.0,90.619624,1141,7042
2,EPA,110,40.81600,-73.90200,2017,1,6.758297,EPA_110,POINT (592600.643 4518911.64),24205,NaN,31,257404650.2,137632001.2,2275138358.9,154688.6,49821436.6,89.852751,1520,9106
3,EPA,133,40.86790,-73.87809,2017,1,8.790909,EPA_133,POINT (594543.361 4524698.573),24205,79564.0,13,610033592.6,535832143.3,6214874085.5,418827.3,89041366.0,101.311307,1221,7042
4,EPA,110,40.81600,-73.90200,2018,1,7.876813,EPA_110,POINT (592600.643 4518911.64),24217,NaN,34,286764302.1,128051600.2,1975416978.0,138749.2,41538383.0,92.904188,1652,9106
5,EPA,133,40.86790,-73.87809,2018,1,11.110000,EPA_133,POINT (594543.361 4524698.573),24217,78297.0,15,539155900.8,355994987.9,3479442596.1,256791.7,56092865.0,91.131356,1279,7042
6,EPA,110,40.81600,-73.90200,2019,1,7.226107,EPA_110,POINT (592600.643 4518911.64),24229,NaN,26,333728602.9,144213239.1,2204097482.2,154745.4,48223570.0,91.421101,1516,9106
7,EPA,133,40.86790,-73.87809,2019,1,7.420000,EPA_133,POINT (594543.361 4524698.573),24229,NaN,17,847169757.5,373639935.8,4590611094.2,339935.7,64957140.6,96.410327,1002,7042
8,EPA,110,40.81600,-73.90200,2020,1,8.254894,EPA_110,POINT (592600.643 4518911.64),24241,NaN,39,297775543.7,125404392.4,2077594609.4,142057.9,48407513.0,87.09542,1466,9106
9,EPA,133,40.86790,-73.87809,2020,1,7.245455,EPA_133,POINT (594543.361 4524698.573),24241,NaN,28,586652406.4,344441835.6,3224369293.8,241091.2,58295578.0,88.086577,925,7042


In [18]:
final_df.to_csv('final_data.csv', index=False)